# TP 0 — Probabilités et statistiques par la simulation
## Remise à niveau : vérifier les rappels avec du code

| | |
|---|---|
| **École** | ESAIP |
| **Classe** | Aix IR 5 CP |
| **Cours** | Détection d’anomalies et fraude avec Python |
| **Enseignant** | Serigne Faye |

## Contexte

Ce TP accompagne le document de rappels de probabilités et de statistiques. Plutôt que d’admettre les résultats, vous allez les **vérifier par simulation** (Monte-Carlo) et les **démontrer numériquement** : théorème de Bayes, robustesse des indicateurs, lois usuelles, distance de Mahalanobis, intervalles de confiance, courbe ROC et seuil optimal.

**Consignes**

- Écrivez du code **vectorisé** (NumPy) : évitez les boucles Python sur des tableaux de 10⁶ éléments.
- Fixez les graines aléatoires (`default_rng(graine)`) pour que vos résultats soient reproductibles.
- Les valeurs simulées varient légèrement d’un tirage à l’autre : comparez-les à la théorie à quelques pour cent près, pas au chiffre exact.

**Bibliothèques** : `numpy`, `pandas`, `matplotlib`, `scipy`. Pour comparer à une implémentation de référence à l’étape 6, `scikit-learn` est facultatif.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

## Étape 1 — Bayes par simulation : la précision d’un détecteur

Un détecteur repère **90 %** des fraudes (rappel) et lève une fausse alerte sur **1 %** des transactions normales. La prévalence de la fraude est de **0,17 %**.

**1a.** Simulez 1 000 000 de transactions et mesurez la précision et l’accuracy.

In [ ]:
rng = np.random.default_rng(0)
pi, tpr, fpr = 0.0017, 0.90, 0.01     # prévalence, rappel, taux de fausses alertes
N = 1_000_000

fraude = ...      # tableau booléen de taille N : True avec la probabilité pi
alerte = ...      # True avec la probabilité tpr si fraude, avec la probabilité fpr sinon (indice : np.where)

tp = ...
fp = ...
fn = ...
tn = ...
precision_emp = ...
accuracy = ...
print(f'TP={tp}  FP={fp}  FN={fn}  TN={tn}')
print(f'Précision empirique = {precision_emp:.4f}   Accuracy = {accuracy:.4f}')

**1b.** Écrivez la précision **théorique** avec le théorème de Bayes et comparez-la à la valeur simulée. Déduisez-en la prévalence minimale pour que la précision atteigne 50 %, puis vérifiez numériquement votre formule pour trois valeurs de `fpr`.

In [ ]:
def precision_theorique(pi, tpr, fpr):
    ...

print('Précision théorique :', round(precision_theorique(pi, tpr, fpr), 4))

def prevalence_pour_precision(p, tpr, fpr):
    # prévalence pi telle que precision_theorique(pi, tpr, fpr) = p  (à dériver)
    ...

for f in [0.001, 0.01, 0.05]:
    pi_50 = prevalence_pour_precision(0.5, tpr, f)
    print(f'fpr = {f:5.3f}  prévalence pour 50 % de précision = {pi_50:.5f}  (vérification : {precision_theorique(pi_50, tpr, f):.3f})')

**1c.** Tracez la précision théorique en fonction de la prévalence (échelle logarithmique de 10⁻⁵ à 0,5) pour trois taux de fausses alertes. Puis comparez trois façons d’améliorer le détecteur pour la prévalence de 0,17 % : diviser `fpr` par 2, monter le rappel à 99 %, diviser `fpr` par 10.

In [ ]:
prev = np.logspace(-5, np.log10(0.5), 200)
plt.figure(figsize=(8, 3.8))
for f in [0.001, 0.01, 0.05]:
    plt.plot(prev, ..., label=f'fpr = {f}')
plt.xscale('log')
plt.xlabel('Prévalence de la fraude')
plt.ylabel('Précision')
plt.legend()
plt.show()

print('fpr / 2   :', ...)
print('rappel 99 % :', ...)
print('fpr / 10  :', ...)

## Étape 2 — Robustesse : combien de valeurs aberrantes tolère un indicateur ?

On part de 10 000 montants log-normaux « normaux », puis on remplace une fraction ε d’entre eux par la valeur aberrante 4 000 €.

**2a.** Pour chaque ε, calculez la moyenne, l’écart-type, la médiane, la MAD et l’IQR, ainsi que le **Z-score** et le **Z-score robuste** de la valeur 4 000 €.

**2b.** Montrez que, pour un contaminant de valeur $v$ dominant fortement la distribution, le Z-score de $v$ vaut approximativement $\sqrt{(1-\varepsilon)/\varepsilon}$. Vérifiez numériquement. À partir de quelle fraction ε le Z-score classique (seuil 3) **cesse-t-il de détecter** la valeur aberrante ?

In [ ]:
rng = np.random.default_rng(1)
base = rng.lognormal(3.3, 0.8, 10_000)

def indicateurs(x):
    # renvoie un dictionnaire : moyenne, ecart_type (ddof=1), mediane, MAD, IQR
    ...

def contaminer(base, eps, valeur=4000):
    # remplace une fraction eps des valeurs par `valeur`
    ...

lignes = []
for eps in [0.001, 0.01, 0.05, 0.10, 0.15, 0.30, 0.45, 0.49, 0.52]:
    x = contaminer(base, eps)
    ind = indicateurs(x)
    z = ...          # Z-score de 4000
    z_rob = ...      # Z-score robuste de 4000 (np.nan si la MAD est nulle)
    lignes.append({'eps': eps, **{k: round(v, 1) for k, v in ind.items()}, 'z': round(z, 2),
                   'z_rob': round(z_rob, 1) if not np.isnan(z_rob) else np.nan,
                   'theorie': round(np.sqrt((1 - eps) / eps), 2)})
tableau = pd.DataFrame(lignes)
tableau

**2c.** Tracez l’évolution des indicateurs de position (moyenne, médiane) et de dispersion (écart-type, MAD, IQR) en fonction de ε (de 0 à 0,6). Repérez le **point de rupture** de chaque indicateur (la fraction de contamination à partir de laquelle il peut prendre une valeur arbitrairement grande).

In [ ]:
epsilons = np.linspace(0, 0.6, 61)
res = pd.DataFrame([indicateurs(contaminer(base, e)) for e in epsilons], index=epsilons)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
...    # gauche : moyenne et médiane en fonction de eps
...    # droite : écart-type, MAD et IQR en fonction de eps
axes[0].set_xlabel('Fraction de contamination ε')
axes[1].set_xlabel('Fraction de contamination ε')
plt.tight_layout()
plt.show()

## Étape 3 — Lois usuelles : Poisson, normale, log-normale et théorème central limite

**3a. Poisson.** Le nombre de transactions d’un client dans une fenêtre de 10 minutes suit approximativement une loi de Poisson de paramètre λ = 0,1. Simulez 1 000 000 de fenêtres, comparez `P(X ≥ k)` empirique et théorique pour k = 1, 2, 3, vérifiez que moyenne et variance sont égales, puis calculez le nombre attendu de **fausses alertes par jour** si la règle « au moins 2 transactions en 10 minutes » est appliquée à 300 clients (144 fenêtres par jour).

In [ ]:
rng = np.random.default_rng(2)
lam = 0.1
comptes = ...      # 1 000 000 tirages de Poisson(lam)

for k in [1, 2, 3]:
    emp = ...      # fréquence empirique de comptes >= k
    theo = ...     # valeur théorique (indice : stats.poisson.sf(k - 1, lam))
    print(f'P(X >= {k}) : empirique = {emp:.6f}   théorique = {theo:.6f}')
print('Moyenne et variance :', comptes.mean().round(4), comptes.var().round(4))
print('Fausses alertes par jour (règle X >= 2, 300 clients, 144 fenêtres) :', ...)
print('Fausses alertes par jour (règle X >= 3) :', ...)

**3b. Normale et log-normale.** Vérifiez empiriquement la règle 68-95-99,7. Puis, pour des montants log-normaux, comparez l’asymétrie (`stats.skew`) de `x` et de `log(x)`, et tracez les diagrammes **quantile-quantile** (`stats.probplot`) de `x` et de `log(x)` contre la loi normale.

In [ ]:
rng = np.random.default_rng(2)
z = rng.normal(size=1_000_000)
for k in [1, 2, 3]:
    print(f'P(|Z| < {k}) : empirique = {...:.4f}   théorique = {...:.4f}')

x = rng.lognormal(3.3, 0.8, 5000)
print('Asymétrie de x :', ..., '| asymétrie de log(x) :', ...)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
...    # QQ-plot de x contre la loi normale (stats.probplot(x, dist='norm', plot=axes[0]))
...    # QQ-plot de log(x)
axes[0].set_title('x')
axes[1].set_title('log(x)')
plt.tight_layout()
plt.show()

**3c. Théorème central limite.** Pour une loi log-normale (σ = 1, très asymétrique), simulez 5 000 moyennes d’échantillons de taille n = 1, 5, 30, 200. Comparez l’asymétrie empirique des moyennes à la valeur théorique $\gamma_1/\sqrt{n}$ où $\gamma_1 = (e+2)\sqrt{e-1}$ est l’asymétrie de la loi log-normale de σ = 1. Expliquez l’écart pour n = 1.

In [ ]:
rng = np.random.default_rng(2)
gamma1 = (np.e + 2) * np.sqrt(np.e - 1)

fig, axes = plt.subplots(1, 4, figsize=(11, 2.8))
for ax, n in zip(axes, [1, 5, 30, 200]):
    moyennes = ...      # 5000 moyennes d'échantillons de taille n d'une loi log-normale(0, 1)
    ax.hist(moyennes, bins=40)
    ax.set_title(f'n = {n}')
    print(f'n = {n:3d}  asymétrie empirique = {...:.2f}   théorique = {...:.2f}')
plt.tight_layout()
plt.show()

## Étape 4 — Covariance, corrélation et distance de Mahalanobis

**4a.** Générez 100 000 points d’une loi normale bivariée de matrice de covariance Σ = [[1 ; 0,8], [0,8 ; 1]] **sans** `multivariate_normal` : utilisez la **factorisation de Cholesky** Σ = L Lᵀ et la transformation X = Z Lᵀ où Z est un bruit gaussien standard. Vérifiez que la covariance empirique est proche de Σ.

**4b.** Montrez qu’un lien fort peut être invisible pour la corrélation : simulez `y = x² + bruit` et calculez la corrélation de Pearson, la corrélation de Spearman et la corrélation entre `x²` et `y`.

In [ ]:
rng = np.random.default_rng(3)
Sigma = np.array([[1.0, 0.8], [0.8, 1.0]])
L = ...                       # factorisation de Cholesky (np.linalg.cholesky)
X = ...                       # 100 000 points : bruit gaussien standard (100000, 2) multiplié par L.T
print('Covariance empirique :')
print(np.cov(X, rowvar=False).round(3))

xx = rng.uniform(-2, 2, 5000)
yy = xx ** 2 + rng.normal(0, 0.2, 5000)
print('Pearson :', ..., '| Spearman :', ..., '| corr(x², y) :', ...)

**4c.** Écrivez une fonction vectorisée `mahalanobis2(X, mu, cov)` qui renvoie D² pour chaque ligne de X. Vérifiez que les quantiles de D² coïncident avec ceux de la loi du χ² à 2 degrés de liberté, et que la part de points au-dessus du seuil du χ² à 99,7 % vaut environ 0,3 %.

**4d.** Les paramètres μ et Σ sont **estimés** sur un échantillon de taille n. Mesurez l’effet de n sur le taux de fausses alertes (part de nouveaux points normaux dépassant le seuil du χ² à 99,7 %) pour n = 20, 50, 200, 5 000 (30 répétitions chacun).

In [ ]:
def mahalanobis2(X, mu, cov):
    ...

d2 = mahalanobis2(X, X.mean(axis=0), np.cov(X, rowvar=False))
for q in [0.5, 0.9, 0.99, 0.997]:
    print(f'quantile {q} : D² empirique = {np.quantile(d2, q):.2f}   χ²(2) = {stats.chi2.ppf(q, 2):.2f}')

seuil = stats.chi2.ppf(0.997, 2)
print(f'Seuil = {seuil:.2f}   part au-dessus : {(d2 > seuil).mean():.4f}')

In [ ]:
for n in [20, 50, 200, 5000]:
    taux = []
    for _ in range(30):
        ech = rng.normal(size=(n, 2)) @ L.T                    # échantillon d'apprentissage
        nouveaux = rng.normal(size=(20_000, 2)) @ L.T          # nouveaux points normaux
        d2_nouveaux = ...                                      # D² avec μ et Σ estimés sur `ech`
        taux.append(...)                                       # part de points au-dessus du seuil
    print(f'n = {n:5d}  taux de fausses alertes moyen = {np.mean(taux):.4f}   (écart-type {np.std(taux):.4f})')

## Étape 5 — Intervalles de confiance d’une précision

Une règle lève 20 alertes, dont 8 sont de vraies fraudes : p̂ = 0,4.

**5a.** Calculez trois intervalles de confiance à 95 % : l’intervalle **approché** (de Wald) $\hat p \pm 1{,}96\sqrt{\hat p(1-\hat p)/n}$, l’intervalle de **Wilson** et un intervalle **bootstrap** (10 000 rééchantillonnages, percentiles 2,5 % et 97,5 %).

Intervalle de Wilson : centre $\dfrac{\hat p + z^2/(2n)}{1 + z^2/n}$, demi-largeur $\dfrac{z}{1 + z^2/n}\sqrt{\dfrac{\hat p(1-\hat p)}{n} + \dfrac{z^2}{4n^2}}$.

In [ ]:
rng = np.random.default_rng(4)
z95 = 1.96

def wald_ci(p_hat, n):
    ...         # renvoie (borne basse, borne haute)

def wilson_ci(p_hat, n):
    ...

k, n = 8, 20
observations = np.array([1] * k + [0] * (n - k))
boot = ...      # 10 000 moyennes de rééchantillons (avec remise) de `observations`

print('Wald      :', np.round(wald_ci(k / n, n), 3))
print('Wilson    :', np.round(wilson_ci(k / n, n), 3))
print('Bootstrap :', np.round(np.quantile(boot, [0.025, 0.975]), 3))

**5b.** Un intervalle « à 95 % » contient-il vraiment le vrai paramètre 95 % du temps ? Mesurez la **couverture** des intervalles de Wald et de Wilson pour une proportion rare p = 1 %, avec n = 50, 200, 1 000, 5 000 (20 000 échantillons simulés pour chaque n).

In [ ]:
p = 0.01
for n in [50, 200, 1000, 5000]:
    p_hat = ...                                   # 20 000 proportions observées (binomiale)
    bas_w, haut_w = wald_ci(p_hat, n)
    bas_s, haut_s = wilson_ci(p_hat, n)
    couv_wald = ...                               # part des intervalles qui contiennent p
    couv_wilson = ...
    print(f'n = {n:5d}  couverture Wald = {couv_wald:.3f}   Wilson = {couv_wilson:.3f}')

## Étape 6 — Courbe ROC et AUC « à la main »

Les scores des transactions normales suivent N(0 ; 1), ceux des fraudes N(2,5 ; 1).

**6a.** Écrivez `roc(y, s)` qui renvoie les taux de faux positifs et de vrais positifs pour tous les seuils (tri des scores puis sommes cumulées), puis l’aire sous la courbe par la méthode des trapèzes. Comparez à la valeur théorique $\Phi(d/\sqrt{2})$, à une estimation par **paires** (probabilité qu’un score de fraude dépasse un score normal) et, si vous l’avez, à `sklearn.metrics.roc_auc_score`.

**6b.** Montrez que l’AUC **ne dépend pas de la prévalence**, alors que la précision au seuil 1,6 en dépend fortement (prévalences 0,5 ; 0,05 ; 0,0017).

In [ ]:
def roc(y, s):
    # renvoie (fpr, tpr) : tableaux commençant à (0, 0), un point par seuil possible
    ...

def auc_trapeze(fpr_, tpr_):
    ...

rng = np.random.default_rng(5)
def simuler(n, pi, d=2.5):
    y = (rng.random(n) < pi).astype(int)
    s = rng.normal(0, 1, n) + d * y
    return y, s

y, s = simuler(200_000, 0.05)
print('AUC (trapèzes)   :', round(auc_trapeze(*roc(y, s)), 4))
print('AUC (théorique)  :', round(stats.norm.cdf(2.5 / np.sqrt(2)), 4))

s1, s0 = s[y == 1], s[y == 0]
i = rng.integers(0, len(s1), 500_000)
j = rng.integers(0, len(s0), 500_000)
print('AUC (par paires) :', ...)

try:
    from sklearn.metrics import roc_auc_score
    print('AUC (scikit-learn) :', round(roc_auc_score(y, s), 4))
except ImportError:
    pass

In [ ]:
for pi_ in [0.5, 0.05, 0.0017]:
    y, s = simuler(1_000_000 if pi_ < 0.01 else 200_000, pi_)
    auc = ...                       # AUC par la fonction ci-dessus
    alerte = s > 1.6
    precision = ...                 # précision au seuil 1,6
    rappel = ...                    # rappel au seuil 1,6
    print(f'prévalence = {pi_:6.4f}   AUC = {auc:.4f}   précision@1,6 = {precision:.4f}   rappel@1,6 = {rappel:.3f}')

## Étape 7 — Le seuil qui minimise le coût

Même modèle de scores (écart d = 2,5 entre les moyennes, variance 1), prévalence π = 0,17 %, coût d’une fraude manquée c_FN = 120 €, coût d’une fausse alerte c_FP = 3 €.

**7a.** Exprimez le coût moyen par transaction en fonction du seuil t : $c_{FN}\,\pi\,(1 - TPR(t)) + c_{FP}\,(1-\pi)\,FPR(t)$, avec TPR(t) = 1 − Φ(t − d) et FPR(t) = 1 − Φ(t). Trouvez le seuil optimal par recherche sur une grille.

**7b.** Démontrez que le seuil optimal vérifie $\text{rapport de vraisemblance}(t^*) = \dfrac{c_{FP}(1-\pi)}{c_{FN}\,\pi}$, et que pour deux gaussiennes de même variance, $t^* = \dfrac{d}{2} + \dfrac{1}{d}\ln\dfrac{c_{FP}(1-\pi)}{c_{FN}\,\pi}$. Comparez à la grille.

**7c.** Comparez le coût (pour 1 000 000 de transactions) au seuil optimal, au seuil qui maximise le **F1**, au seuil 1,6 et à l’absence de détection.

In [ ]:
d, pi, c_fn, c_fp = 2.5, 0.0017, 120, 3
seuils = np.linspace(-1, 6, 7001)
tpr_t = stats.norm.sf(seuils - d)
fpr_t = stats.norm.sf(seuils)

cout = ...                                  # coût moyen par transaction pour chaque seuil
t_opt = ...                                 # seuil qui minimise le coût (grille)
t_formule = ...                             # seuil donné par la formule de la question 7b
print(f'Seuil optimal (grille) = {t_opt:.3f}   (formule : {t_formule:.3f})')

precision = tpr_t * pi / (tpr_t * pi + fpr_t * (1 - pi))
f1 = 2 * precision * tpr_t / (precision + tpr_t)
t_f1 = ...                                  # seuil qui maximise le F1 (attention aux NaN : np.nanargmax)

def cout_pour(t):
    return ...                              # coût moyen par transaction au seuil t

for nom, t in [('optimal (coût)', t_opt), ('maximisant le F1', t_f1), ('1,6', 1.6)]:
    print(f'Seuil {nom:17s} t = {t:.3f}   coût pour 1 000 000 transactions = {cout_pour(t) * 1e6:,.0f} €')
print(f'Aucune détection : {c_fn * pi * 1e6:,.0f} €')

**7d.** Tracez le coût en fonction du seuil, et marquez le seuil optimal et le seuil maximisant le F1. Donnez la précision et le rappel au seuil optimal.

In [ ]:
plt.figure(figsize=(8, 3.6))
...
plt.xlabel('Seuil t')
plt.ylabel('Coût moyen par transaction (€)')
plt.legend()
plt.show()

i_opt = ...      # indice du seuil optimal
print(f'Au seuil optimal : précision = {precision[i_opt]:.3f}   rappel = {tpr_t[i_opt]:.3f}')

## Questions de synthèse

Répondez dans les cellules de texte situées sous chaque question, en vous appuyant sur vos résultats.

**Question 1 (Bayes).** Comparez la précision simulée et la précision théorique. Quelle prévalence faut-il pour atteindre 50 % de précision avec ce détecteur ? Parmi les trois améliorations testées, laquelle est efficace et pourquoi ?

*Votre réponse :*

**Question 2 (robustesse).** À partir de quelle fraction de contamination le Z-score cesse-t-il de détecter la valeur 4 000 € ? Justifiez par la formule. Donnez le point de rupture approximatif de la moyenne et de l’écart-type, de l’IQR, de la médiane et de la MAD d’après vos courbes.

*Votre réponse :*

**Question 3 (lois).** Sous l’hypothèse de Poisson, combien de fausses alertes par jour produisent les règles « au moins 2 » et « au moins 3 » transactions en 10 minutes ? Qu’en concluez-vous sur le choix du seuil ? Pourquoi faut-il passer au logarithme pour appliquer un Z-score à des montants ? Que dit le théorème central limite sur les moyennes, et pourquoi l’asymétrie empirique pour n = 1 est-elle inférieure à la valeur théorique ?

*Votre réponse :*

**Question 4 (Mahalanobis).** Les quantiles de D² suivent-ils la loi du χ² ? Pourquoi la corrélation de Pearson est-elle presque nulle pour `y = x²` alors que le lien est parfait ? Quel est l’effet de la taille d’échantillon n sur le taux de fausses alertes de la distance de Mahalanobis ? Que recommandez-vous en pratique ?

*Votre réponse :*

**Question 5 (intervalles).** Comparez les trois intervalles pour 8 fraudes sur 20 alertes. Que montre l’étude de couverture pour une proportion rare ? Quel intervalle utiliseriez-vous pour la précision d’une règle qui lève 20 alertes ?

*Votre réponse :*

**Question 6 (ROC et coût).** Pourquoi l’AUC est-elle stable quand la prévalence change alors que la précision s’effondre ? Quel est le seuil de coût minimal, et pourquoi diffère-t-il du seuil qui maximise le F1 ? Quelle est la précision de la banque au seuil optimal et pourquoi est-elle si faible sans que ce soit un problème ?

*Votre réponse :*

## Pour aller plus loin (facultatif)

1. **Bayes séquentiel.** Deux détecteurs indépendants (rappel 0,81 et taux de fausses alertes 0,055 chacun) alertent tous les deux. Calculez la précision de la règle « les deux alertent » pour π = 0,17 %, par la formule puis par simulation. Montrez que les *odds* a posteriori se multiplient par le rapport de vraisemblance de chaque détecteur.
2. **Variabilité d’une précision mesurée.** Pour un test contenant seulement 36 fraudes, simulez 5 000 tests et mesurez l’écart-type du rappel observé. Que dit-il sur la comparaison de deux modèles dont les rappels diffèrent de 3 points ?
3. **Plus de deux variables.** Reprenez la distance de Mahalanobis en dimension p = 5 et p = 20 : quel est le seuil du χ² à 99,7 % et quelle taille d’échantillon faut-il pour que le taux de fausses alertes reste proche de 0,3 % ?